In [1]:
import sys
import os

# Try to downgrade numpy if needed
try:
    import numpy as np
    if np.__version__.startswith('2'):
        print(f"NumPy version {np.__version__} detected - attempting workaround...")
        # Set environment variables to avoid numexpr issues
        os.environ['NUMEXPR_MAX_THREADS'] = '1'
        os.environ['USE_NUMEXPR'] = '0'
except:
    pass

# Import with modified behavior
import pandas._config.config as cf
cf.options.compute.use_numexpr = False

# Now do your imports
import pandas as pd
import numpy as np
from typing import List, Dict, Tuple, Optional
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

print("Imports successful!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

NumPy version 2.2.6 detected - attempting workaround...



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Users\Luffy\anaconda3\envs\new_pytorch_env\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\Luffy\anaconda3\envs\new_pytorch_env\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "C:\Users\Luffy\anaconda3\envs\new_pytorch_env\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\Luffy\anaconda3\envs\new_pytorch_env\lib\site-packages\traitlets\config\application.py", line 10

AttributeError: _ARRAY_API not found

Imports successful!
Pandas version: 2.3.3
NumPy version: 2.2.6


In [2]:
class OptimizedDataPreprocessor:
    """Оптимизированный препроцессинг данных"""
    
    def __init__(self):
        self.scaler = StandardScaler()
        self.feature_columns = None
        
    def create_temporal_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """Создание временных признаков (оптимизированная версия)"""
        df_processed = df.copy()
        
        # Извлечение даты и времени
        if 'Date_Time' in df_processed.columns:
            dt = pd.to_datetime(df_processed['Date_Time'])
        elif 'datetime' in df_processed.columns:
            dt = pd.to_datetime(df_processed['datetime'])
        elif 'timestamp' in df_processed.columns:
            dt = pd.to_datetime(df_processed['timestamp'])
        else:
            # Если нет колонки с датой/временем, используем индекс
            if isinstance(df_processed.index, pd.DatetimeIndex):
                dt = df_processed.index
            else:
                raise ValueError("Не найдена колонка с датой/временем")
        
        # Базовые временные признаки
        df_processed['hour'] = dt.dt.hour.astype('int8')
        df_processed['day_of_week'] = dt.dt.dayofweek.astype('int8')
        df_processed['month'] = dt.dt.month.astype('int8')
        df_processed['day_of_year'] = dt.dt.dayofyear.astype('int16')
        
        # Циклические признаки (оптимизировано)
        hour_rad = 2 * np.pi * df_processed['hour'] / 24
        df_processed['hour_sin'] = np.sin(hour_rad).astype('float32')
        df_processed['hour_cos'] = np.cos(hour_rad).astype('float32')
        
        dow_rad = 2 * np.pi * df_processed['day_of_week'] / 7
        df_processed['dow_sin'] = np.sin(dow_rad).astype('float32')
        df_processed['dow_cos'] = np.cos(dow_rad).astype('float32')
        
        # Бинарные признаки
        df_processed['is_weekend'] = (df_processed['day_of_week'] >= 5).astype('int8')
        df_processed['is_night'] = ((df_processed['hour'] >= 22) | 
                                   (df_processed['hour'] <= 6)).astype('int8')
        df_processed['is_working_hours'] = (
            (df_processed['hour'] >= 8) & 
            (df_processed['hour'] <= 18) & 
            (df_processed['is_weekend'] == 0)
        ).astype('int8')
        
        return df_processed
    
    def resample_to_hour(self, df: pd.DataFrame, sensor_cols: List[str]) -> pd.DataFrame:
        """Агрегация данных до 1-часовых интервалов"""
        df_resampled = df.copy()
        
        # Определяем колонку с датой/временем
        time_col = None
        if 'Date_Time' in df_resampled.columns:
            time_col = 'Date_Time'
        elif 'datetime' in df_resampled.columns:
            time_col = 'datetime'
        elif 'timestamp' in df_resampled.columns:
            time_col = 'timestamp'
        else:
            if isinstance(df_resampled.index, pd.DatetimeIndex):
                df_resampled.reset_index(inplace=True)
                if df_resampled.columns[0] in ['index', 'Date_Time', 'datetime', 'timestamp']:
                    time_col = df_resampled.columns[0]
        
        if time_col and time_col in df_resampled.columns:
            df_resampled.set_index(time_col, inplace=True)
        elif isinstance(df_resampled.index, pd.DatetimeIndex):
            # Индекс уже является DatetimeIndex
            pass
        else:
            raise ValueError("DataFrame должен содержать колонку 'Date_Time' или иметь DatetimeIndex")
        
        # Определение типов колонок
        numeric_cols = [col for col in sensor_cols 
                       if col in df_resampled.columns and 
                       df_resampled[col].dtype in ['float32', 'float64', 'int32', 'int64']]
        
        # Ресемплинг числовых данных
        if numeric_cols:
            numeric_resampled = df_resampled[numeric_cols].resample('1H').mean()
        else:
            numeric_resampled = pd.DataFrame()
        
        # Ресемплинг временных признаков
        temporal_cols = ['hour', 'day_of_week', 'month', 'day_of_year',
                        'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
                        'is_weekend', 'is_night', 'is_working_hours']
        
        temporal_cols = [col for col in temporal_cols if col in df_resampled.columns]
        
        if temporal_cols:
            # Для бинарных и категориальных признаков используем моду или первое значение
            temporal_resampled = df_resampled[temporal_cols].resample('1H').agg(
                lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0] if len(x) > 0 else 0
            )
        else:
            temporal_resampled = pd.DataFrame()
        
        # Объединение результатов
        if not numeric_resampled.empty and not temporal_resampled.empty:
            df_1hour = pd.concat([numeric_resampled, temporal_resampled], axis=1)
        elif not numeric_resampled.empty:
            df_1hour = numeric_resampled
        elif not temporal_resampled.empty:
            df_1hour = temporal_resampled
        else:
            df_1hour = pd.DataFrame()
        
        # Заполнение пропусков
        if not df_1hour.empty:
            df_1hour = df_1hour.ffill().bfill()
            
            # Приведение типов
            for col in df_1hour.columns:
                if col.startswith('is_'):
                    df_1hour[col] = df_1hour[col].fillna(0).round().astype('int8')
                elif col in ['hour', 'day_of_week', 'month']:
                    df_1hour[col] = df_1hour[col].fillna(0).round().astype('int8')
                elif col in ['day_of_year']:
                    df_1hour[col] = df_1hour[col].fillna(0).round().astype('int16')
        
        return df_1hour.reset_index()
    
    def prepare_features(self, df: pd.DataFrame, target_col: str = None) -> Tuple[np.ndarray, Optional[np.ndarray]]:
        """Подготовка признаков для модели"""
        # Выбор числовых колонок
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        
        if target_col and target_col in numeric_cols:
            numeric_cols.remove(target_col)
        
        self.feature_columns = numeric_cols
        
        # Масштабирование признаков
        X = df[numeric_cols].values
        X_scaled = self.scaler.fit_transform(X)
        
        # Подготовка целевой переменной
        y = None
        if target_col and target_col in df.columns:
            y = df[target_col].values
        
        return X_scaled, y


In [3]:
class OptimizedNeuralNetwork(nn.Module):
    """Оптимизированная нейросеть для анализа данных"""
    
    def __init__(self, input_size: int, hidden_dims: List[int] = [64, 32, 16]):
        super().__init__()
        
        layers = []
        prev_dim = input_size
        
        # Создание скрытых слоев
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.2)
            ])
            prev_dim = hidden_dim
        
        # Выходной слой
        layers.append(nn.Linear(prev_dim, 1))
        
        self.model = nn.Sequential(*layers)
        
        # Инициализация весов
        self._init_weights()
    
    def _init_weights(self):
        """Инициализация весов сети"""
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.kaiming_normal_(module.weight, mode='fan_out', nonlinearity='relu')
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0)
    
    def forward(self, x):
        """Прямой проход"""
        return self.model(x).squeeze()

In [4]:
class SensorAnalyzer:
    """Основной класс для анализа данных с датчиков"""
    
    def __init__(self, device: str = 'cuda' if torch.cuda.is_available() else 'cpu'):
        self.device = torch.device(device)
        self.preprocessor = OptimizedDataPreprocessor()
        self.model = None
        self.is_trained = False
    
    def identify_sensor_columns(self, df: pd.DataFrame) -> List[str]:
        """Автоматическое определение колонок с датчиками"""
        # Исключаем колонки, которые не являются датчиками
        exclude_keywords = {'date', 'time', 'datetime', 'timestamp', 'target', 'id', 'index'}
        
        sensor_cols = []
        for col in df.columns:
            col_lower = col.lower()
            # Проверяем, не является ли колонка временной меткой
            if not any(keyword in col_lower for keyword in exclude_keywords):
                # Проверяем, является ли колонка числовой
                if pd.api.types.is_numeric_dtype(df[col]):
                    sensor_cols.append(col)
        
        return sensor_cols
    
    def load_and_preprocess(self, filepath: str, sensor_cols: List[str] = None) -> pd.DataFrame:
        """Загрузка и предобработка данных"""
        # Загрузка данных
        try:
            df = pd.read_csv(filepath, parse_dates=['Date_Time', 'datetime', 'timestamp'])
        except:
            try:
                df = pd.read_excel(filepath, parse_dates=['Date_Time', 'datetime', 'timestamp'])
            except:
                # Если не удается распознать формат, пробуем без указания колонок даты
                df = pd.read_csv(filepath)
        
        # Если не указаны колонки датчиков, определяем их автоматически
        if sensor_cols is None:
            sensor_cols = self.identify_sensor_columns(df)
            print(f"Автоматически определены колонки датчиков: {sensor_cols}")
        
        # Создание временных признаков
        df = self.preprocessor.create_temporal_features(df)
        
        # Агрегация до часовых интервалов
        df_hourly = self.preprocessor.resample_to_hour(df, sensor_cols)
        
        return df_hourly
    
    def train_model(self, X: np.ndarray, y: np.ndarray, 
                   test_size: float = 0.2, epochs: int = 50,
                   batch_size: int = 32) -> Dict:
        """Обучение модели"""
        # Разделение данных
        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size=test_size, random_state=42
        )
        
        # Преобразование в тензоры
        X_train_tensor = torch.FloatTensor(X_train).to(self.device)
        y_train_tensor = torch.FloatTensor(y_train).to(self.device)
        X_val_tensor = torch.FloatTensor(X_val).to(self.device)
        y_val_tensor = torch.FloatTensor(y_val).to(self.device)
        
        # Создание даталоадеров
        train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
        val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
        
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
        
        # Инициализация модели
        input_size = X_train.shape[1]
        self.model = OptimizedNeuralNetwork(input_size).to(self.device)
        
        # Оптимизатор и функция потерь
        optimizer = optim.Adam(self.model.parameters(), lr=0.001)
        criterion = nn.BCEWithLogitsLoss()
        
        # Обучение
        train_losses = []
        val_losses = []
        best_val_loss = float('inf')
        patience = 10
        patience_counter = 0
        
        for epoch in range(epochs):
            # Фаза обучения
            self.model.train()
            train_loss = 0.0
            
            for batch_X, batch_y in train_loader:
                optimizer.zero_grad()
                outputs = self.model(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()
            
            avg_train_loss = train_loss / len(train_loader)
            train_losses.append(avg_train_loss)
            
            # Фаза валидации
            self.model.eval()
            val_loss = 0.0
            
            with torch.no_grad():
                for batch_X, batch_y in val_loader:
                    outputs = self.model(batch_X)
                    val_loss += criterion(outputs, batch_y).item()
            
            avg_val_loss = val_loss / len(val_loader)
            val_losses.append(avg_val_loss)
            
            # Ранняя остановка
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                patience_counter = 0
                torch.save(self.model.state_dict(), 'best_model.pth')
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    break
        
        # Загрузка лучшей модели
        self.model.load_state_dict(torch.load('best_model.pth'))
        self.is_trained = True
        
        return {
            'train_losses': train_losses,
            'val_losses': val_losses,
            'best_val_loss': best_val_loss,
            'epochs_trained': epoch + 1
        }
    
    def predict(self, X: np.ndarray) -> np.ndarray:
        """Предсказание на новых данных"""
        if not self.is_trained:
            raise ValueError("Модель не обучена. Сначала вызовите train_model()")
        
        self.model.eval()
        X_tensor = torch.FloatTensor(X).to(self.device)
        
        with torch.no_grad():
            predictions = self.model(X_tensor)
        
        return torch.sigmoid(predictions).cpu().numpy()
    
    def analyze_stops(self, df: pd.DataFrame, sensor_col: str, 
                     threshold: float = 0.5, min_duration: int = 3) -> List[Dict]:
        """Анализ остановок оборудования"""
        if not self.is_trained:
            raise ValueError("Модель не обучена")
        
        # Подготовка признаков
        X, _ = self.preprocessor.prepare_features(df)
        
        # Предсказание
        predictions = self.predict(X)
        
        # Поиск интервалов остановок
        stops = []
        in_stop = False
        start_time = None
        
        for i, (pred, time) in enumerate(zip(predictions, df['Date_Time'])):
            if pred >= threshold and not in_stop:
                in_stop = True
                start_time = time
            elif pred < threshold and in_stop:
                in_stop = False
                stop_duration = (time - start_time).total_seconds() / 3600
                
                if stop_duration >= min_duration:
                    stops.append({
                        'start_time': start_time,
                        'end_time': time,
                        'duration_hours': stop_duration,
                        'sensor': sensor_col
                    })
        
        return stops

In [5]:
def find_common_stops(stops_dict: Dict[str, List[Dict]], 
                     min_overlap: int = 3) -> List[Dict]:
    """Поиск общих остановок на нескольких датчиках"""
    all_stops = []
    
    # Сбор всех остановок
    for sensor, stops in stops_dict.items():
        for stop in stops:
            all_stops.append({
                'sensor': sensor,
                'start': stop['start_time'],
                'end': stop['end_time']
            })
    
    # Сортировка по времени начала
    all_stops.sort(key=lambda x: x['start'])
    
    overlapping_stops = []
    n = len(all_stops)
    
    # Поиск пересечений
    for i in range(n):
        current = all_stops[i]
        overlapping_sensors = {current['sensor']}
        overlap_start = current['start']
        overlap_end = current['end']
        
        for j in range(i + 1, n):
            other = all_stops[j]
            
            # Проверка пересечения
            if other['start'] <= overlap_end:
                overlapping_sensors.add(other['sensor'])
                overlap_start = max(overlap_start, other['start'])
                overlap_end = min(overlap_end, other['end'])
            else:
                break
        
        # Проверка минимального количества пересекающихся датчиков
        if len(overlapping_sensors) >= min_overlap:
            overlapping_stops.append({
                'sensors': list(overlapping_sensors),
                'start_time': overlap_start,
                'end_time': overlap_end,
                'duration_hours': (overlap_end - overlap_start).total_seconds() / 3600,
                'sensor_count': len(overlapping_sensors)
            })
    
    return overlapping_stops


# Пример использования с файлом temperature_data.csv
def main():
    """Пример использования оптимизированного кода"""
    
    # Инициализация анализатора
    analyzer = SensorAnalyzer()
    
    # Путь к файлу данных
    file_path = r"C:\Users\Luffy\Downloads\тетрадки юпитер\temperature_data.csv"
    
    # Конкретные названия датчиков
    sensor_columns = [
        '10HAH01CT103', '10HAH01CT102',
        '10HAH12CT110', '10HAH12CT108', '10HAH12CT106',
        '10HAH11CT114', '10HAH11CT113'
    ]
    
    # Загрузка и предобработка данных с конкретными датчиками
    print("Загрузка данных...")
    df_processed = analyzer.load_and_preprocess(file_path, sensor_cols=sensor_columns)
    
    # Подготовка признаков (предполагаем, что целевая колонка называется 'target' или 'Target')
    target_col = None
    possible_target_cols = ['target', 'Target', 'TARGET', 'label', 'Label', 'LABEL']
    for col in possible_target_cols:
        if col in df_processed.columns:
            target_col = col
            break
    
    print("Подготовка признаков...")
    X, y = analyzer.preprocessor.prepare_features(df_processed, target_col=target_col)
    
    # Если целевая переменная не найдена, создаем искусственную задачу
    if y is None:
        print("Целевая переменная не найдена, создаем искусственную задачу...")
        # Создаем бинарную классификацию на основе одного из признаков
        if len(X) > 0:
            y = (X[:, 0] > np.median(X[:, 0])).astype(float)  # Используем первый признак для создания целевой переменной
    
    # Обучение модели
    print("Обучение модели...")
    training_history = analyzer.train_model(X, y, epochs=30)
    
    print(f"Обучение завершено. Лучшая валидационная ошибка: {training_history['best_val_loss']:.4f}")
    print(f"Эпох обучено: {training_history['epochs_trained']}")
    
    # Анализ остановок для первого датчика из списка
    if sensor_columns:
        print(f"\nАнализ остановок оборудования для датчика {sensor_columns[0]}...")
        stops = analyzer.analyze_stops(df_processed, sensor_columns[0])
        
        print(f"Найдено остановок: {len(stops)}")
        for stop in stops[:5]:  # Показать первые 5 остановок
            print(f"  - {stop['start_time']} до {stop['end_time']} "
                  f"({stop['duration_hours']:.1f} часов)")
    
    return analyzer, df_processed


if __name__ == "__main__":
    main()

Загрузка данных...


TypeError: Only valid with DatetimeIndex, TimedeltaIndex or PeriodIndex, but got an instance of 'Index'